# 📊 Dia 2 - Storytelling com Dados

## 🎯 Objetivo
Contar histórias através dos dados usando visualizações interativas!

## 📖 Perguntas que vamos responder:
1. Qual senador mais gastou em 2022?
2. Quais os tipos de despesas mais comuns?
3. Como foi a evolução dos gastos ao longo do ano?
4. Qual a distribuição de valores das despesas?
5. Quais fornecedores receberam mais?

## 1️⃣ Importar Bibliotecas

In [1]:
# Bibliotecas básicas
import pandas as pd
import numpy as np

# Biblioteca de visualização INTERATIVA
import plotly.express as px
import plotly.graph_objects as go

print("✅ Bibliotecas importadas!")
print("📊 Plotly instalado para gráficos interativos!")

✅ Bibliotecas importadas!
📊 Plotly instalado para gráficos interativos!


## 2️⃣ Carregar e Preparar Dados

In [2]:
# Carregar dados
df = pd.read_csv('../Dados Abertos - CEAPS - csv/despesa_ceaps_2022.csv', 
                 encoding='latin-1', 
                 sep=';',
                 skiprows=1)

# Limpar nomes das colunas
df.columns = df.columns.str.strip().str.replace('"', '')

# Converter valor para numérico (trocar vírgula por ponto)
df['VALOR'] = df['VALOR_REEMBOLSADO'].str.replace(',', '.').astype(float)

# Converter data
df['DATA'] = pd.to_datetime(df['DATA'], format='%d/%m/%Y', errors='coerce')

print(f"✅ Dados carregados: {len(df):,} despesas")
print(f"💰 Valor total: R$ {df['VALOR'].sum():,.2f}")
print(f"👥 Total de senadores: {df['SENADOR'].nunique()}")

✅ Dados carregados: 16,805 despesas
💰 Valor total: R$ 27,323,316.33
👥 Total de senadores: 97


## 📈 HISTÓRIA 1: Quem mais gastou em 2022?

In [3]:
# Calcular total gasto por senador
gastos_senador = df.groupby('SENADOR')['VALOR'].sum().sort_values(ascending=False).head(15)

print("💰 Top 15 Senadores que MAIS GASTARAM em 2022:")
print("="*60)
for i, (senador, valor) in enumerate(gastos_senador.items(), 1):
    print(f"{i:2d}. {senador:30s} R$ {valor:>12,.2f}")
print("="*60)

💰 Top 15 Senadores que MAIS GASTARAM em 2022:
 1. LUCAS BARRETO                  R$   511,319.78
 2. DAVI ALCOLUMBRE                R$   505,054.54
 3. OMAR AZIZ                      R$   494,369.33
 4. TELMÁRIO MOTA                  R$   488,693.40
 5. MECIAS DE JESUS                R$   488,586.66
 6. ROGÉRIO CARVALHO               R$   487,764.65
 7. RANDOLFE RODRIGUES             R$   487,758.92
 8. CHICO RODRIGUES                R$   486,958.05
 9. MAILZA GOMES                   R$   465,899.61
10. ELMANO FÉRRER                  R$   465,194.93
11. ELIZIANE GAMA                  R$   451,042.80
12. PAULO ROCHA                    R$   442,011.34
13. ROBERTO ROCHA                  R$   441,234.58
14. MARCELO CASTRO                 R$   435,789.13
15. FERNANDO BEZERRA COELHO        R$   435,199.20


In [4]:
# GRÁFICO INTERATIVO - Top 15 Senadores
fig = px.bar(x=gastos_senador.values, 
             y=gastos_senador.index,
             orientation='h',
             title='💰 Top 15 Senadores que Mais Gastaram em 2022',
             labels={'x': 'Valor Total (R$)', 'y': 'Senador'},
             color=gastos_senador.values,
             color_continuous_scale='Reds')

fig.update_layout(height=600, showlegend=False)
fig.show()

print("\n💡 DICA: Passe o mouse sobre as barras para ver os valores exatos!")


💡 DICA: Passe o mouse sobre as barras para ver os valores exatos!


## 📊 HISTÓRIA 2: Tipos de Despesas

In [5]:
# Analisar tipos de despesas
tipos_despesa = df.groupby('TIPO_DESPESA')['VALOR'].agg(['sum', 'count']).sort_values('sum', ascending=False)
tipos_despesa.columns = ['Valor Total', 'Quantidade']

print("📋 Tipos de Despesas - Ranking por Valor:")
print("="*80)
for i, (tipo, row) in enumerate(tipos_despesa.head(10).iterrows(), 1):
    print(f"{i:2d}. {tipo[:50]:50s}")
    print(f"    💰 Total: R$ {row['Valor Total']:>12,.2f} | 📊 Qtd: {row['Quantidade']:>6,.0f}")
    print()

📋 Tipos de Despesas - Ranking por Valor:
 1. Passagens aéreas, aquáticas e terrestres nacionais
    💰 Total: R$ 7,423,646.13 | 📊 Qtd:  3,383

 2. Contratação de consultorias, assessorias, pesquisa
    💰 Total: R$ 6,321,484.03 | 📊 Qtd:  1,043

 3. Locomoção, hospedagem, alimentação, combustíveis e
    💰 Total: R$ 5,259,803.51 | 📊 Qtd:  6,769

 4. Aluguel de imóveis para escritório político, compr
    💰 Total: R$ 3,976,536.92 | 📊 Qtd:  3,322

 5. Divulgação da atividade parlamentar               
    💰 Total: R$ 3,345,915.36 | 📊 Qtd:  1,038

 6. Aquisição de material de consumo para uso no escri
    💰 Total: R$   983,707.82 | 📊 Qtd:  1,226

 7. Serviços de Segurança Privada                     
    💰 Total: R$    12,222.56 | 📊 Qtd:     24



In [6]:
# GRÁFICO DE PIZZA - Distribuição por tipo
top_tipos = tipos_despesa.head(8)

fig = px.pie(values=top_tipos['Valor Total'], 
             names=top_tipos.index,
             title='🥧 Distribuição dos Gastos por Tipo de Despesa (Top 8)',
             hole=0.4)  # Donut chart

fig.update_traces(textposition='inside', textinfo='percent+label')
fig.show()

print("\n💡 Clique nas legendas para mostrar/ocultar categorias!")


💡 Clique nas legendas para mostrar/ocultar categorias!


## 📅 HISTÓRIA 3: Evolução ao Longo do Ano

In [7]:
# Gastos por mês
gastos_mes = df.groupby('MES')['VALOR'].sum().sort_index()

meses_nome = ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun', 
              'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez']

print("📅 Gastos Mensais em 2022:")
print("="*50)
for mes, valor in gastos_mes.items():
    print(f"{meses_nome[int(mes)-1]:3s}: R$ {valor:>12,.2f}")
print("="*50)
print(f"TOTAL: R$ {gastos_mes.sum():>12,.2f}")

📅 Gastos Mensais em 2022:
Jan: R$ 1,814,946.67
Fev: R$ 2,251,776.74
Mar: R$ 2,735,146.17
Abr: R$ 2,416,976.89
Mai: R$ 2,573,132.09
Jun: R$ 2,223,460.15
Jul: R$ 2,201,189.79
Ago: R$ 2,107,864.03
Set: R$ 1,637,100.87
Out: R$ 2,099,630.31
Nov: R$ 2,474,749.82
Dez: R$ 2,787,342.80
TOTAL: R$ 27,323,316.33


In [8]:
# GRÁFICO DE LINHA - Evolução mensal
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=[meses_nome[int(m)-1] for m in gastos_mes.index],
    y=gastos_mes.values,
    mode='lines+markers',
    name='Gastos',
    line=dict(color='#FF6B6B', width=3),
    marker=dict(size=10)
))

fig.update_layout(
    title='📈 Evolução dos Gastos ao Longo de 2022',
    xaxis_title='Mês',
    yaxis_title='Valor Total (R$)',
    hovermode='x unified',
    height=500
)

fig.show()

print("\n💡 Passe o mouse sobre os pontos para ver os valores!")


💡 Passe o mouse sobre os pontos para ver os valores!


## 💵 HISTÓRIA 4: Distribuição de Valores

In [9]:
# Estatísticas dos valores
print("💵 Estatísticas das Despesas:")
print("="*50)
print(f"Menor despesa:  R$ {df['VALOR'].min():>12,.2f}")
print(f"Maior despesa:  R$ {df['VALOR'].max():>12,.2f}")
print(f"Média:          R$ {df['VALOR'].mean():>12,.2f}")
print(f"Mediana:        R$ {df['VALOR'].median():>12,.2f}")
print("="*50)

# Faixas de valores
print("\n📊 Distribuição por Faixa de Valor:")
faixas = [
    (0, 500, 'Até R$ 500'),
    (500, 1000, 'R$ 500 - R$ 1.000'),
    (1000, 2000, 'R$ 1.000 - R$ 2.000'),
    (2000, 5000, 'R$ 2.000 - R$ 5.000'),
    (5000, 10000, 'R$ 5.000 - R$ 10.000'),
    (10000, float('inf'), 'Acima de R$ 10.000')
]

for min_val, max_val, label in faixas:
    count = len(df[(df['VALOR'] >= min_val) & (df['VALOR'] < max_val)])
    pct = (count / len(df)) * 100
    print(f"{label:25s}: {count:>6,} despesas ({pct:>5.1f}%)")

💵 Estatísticas das Despesas:
Menor despesa:  R$         0.01
Maior despesa:  R$    77,012.00
Média:          R$     1,625.90
Mediana:        R$       481.44

📊 Distribuição por Faixa de Valor:
Até R$ 500               :  8,508 despesas ( 50.6%)
R$ 500 - R$ 1.000        :  1,670 despesas (  9.9%)
R$ 1.000 - R$ 2.000      :  2,336 despesas ( 13.9%)
R$ 2.000 - R$ 5.000      :  3,168 despesas ( 18.9%)
R$ 5.000 - R$ 10.000     :    731 despesas (  4.3%)
Acima de R$ 10.000       :    392 despesas (  2.3%)


In [10]:
# HISTOGRAMA - Distribuição de valores
fig = px.histogram(df[df['VALOR'] < 10000],  # Filtrar valores muito altos para melhor visualização
                   x='VALOR',
                   nbins=50,
                   title='📊 Distribuição dos Valores das Despesas (até R$ 10.000)',
                   labels={'VALOR': 'Valor da Despesa (R$)', 'count': 'Quantidade'},
                   color_discrete_sequence=['#4ECDC4'])

fig.update_layout(height=500)
fig.show()

print("\n💡 A maioria das despesas está concentrada em valores menores!")


💡 A maioria das despesas está concentrada em valores menores!


## 🏢 HISTÓRIA 5: Principais Fornecedores

In [11]:
# Top fornecedores
top_fornecedores = df.groupby('FORNECEDOR')['VALOR'].sum().sort_values(ascending=False).head(15)

print("🏢 Top 15 Fornecedores que Mais Receberam:")
print("="*70)
for i, (fornecedor, valor) in enumerate(top_fornecedores.items(), 1):
    print(f"{i:2d}. {fornecedor[:40]:40s} R$ {valor:>12,.2f}")
print("="*70)

🏢 Top 15 Fornecedores que Mais Receberam:
 1. ADRIA VIAGENS E TURISMO LTDA             R$ 1,025,684.51
 2. LATAM                                    R$   886,836.74
 3. GOL                                      R$   538,930.03
 4. UPLINK Assessoria e Consultoria Empresar R$   360,000.00
 5. LEONARDO CRUZ SOCIEDADE INDIVIDUAL DE AD R$   343,000.00
 6. Adria Viagens e Turismo Ltda             R$   313,153.42
 7. AZUL                                     R$   300,849.41
 8. BORA COMUNICAÇÃO E MARKETING DIGITAL LTD R$   300,000.00
 9. L COELHO SERRA                           R$   300,000.00
10. LM TURISMO                               R$   284,495.51
11. C Freitas do Nascimento                  R$   220,000.00
12. A. Camacho Torres - VOO Turismo          R$   212,996.66
13. L&L CONSULTORIA LTDA                     R$   202,500.00
14. Adria Viagens e Turismo LTDA ME          R$   201,909.91
15. INOP - INSTITUTO NACIONAL DE ORÇAMENTO P R$   195,000.00


In [12]:
# GRÁFICO DE BARRAS - Top Fornecedores
fig = px.bar(x=top_fornecedores.index, 
             y=top_fornecedores.values,
             title='🏢 Top 15 Fornecedores - Total Recebido em 2022',
             labels={'x': 'Fornecedor', 'y': 'Valor Total (R$)'},
             color=top_fornecedores.values,
             color_continuous_scale='Blues')

fig.update_layout(height=600, xaxis_tickangle=-45, showlegend=False)
fig.show()

print("\n💡 Gire o gráfico e dê zoom para explorar melhor!")


💡 Gire o gráfico e dê zoom para explorar melhor!


## 🎯 RESUMO FINAL - A História Completa

In [13]:
print("="*80)
print("📊 RESUMO EXECUTIVO - CEAPS 2022")
print("="*80)
print()
print(f"💰 VALOR TOTAL GASTO: R$ {df['VALOR'].sum():,.2f}")
print(f"📊 TOTAL DE DESPESAS: {len(df):,}")
print(f"👥 SENADORES ATIVOS: {df['SENADOR'].nunique()}")
print(f"🏢 FORNECEDORES ÚNICOS: {df['FORNECEDOR'].nunique():,}")
print()
print("🔝 DESTAQUES:")
print(f"   • Senador que mais gastou: {gastos_senador.index[0]}")
print(f"     Valor: R$ {gastos_senador.values[0]:,.2f}")
print()
print(f"   • Tipo de despesa mais comum: {tipos_despesa.index[0][:50]}")
print(f"     Total: R$ {tipos_despesa.iloc[0]['Valor Total']:,.2f}")
print()
print(f"   • Mês com maior gasto: {meses_nome[gastos_mes.idxmax()-1]}")
print(f"     Valor: R$ {gastos_mes.max():,.2f}")
print()
print(f"   • Média por despesa: R$ {df['VALOR'].mean():,.2f}")
print("="*80)

📊 RESUMO EXECUTIVO - CEAPS 2022

💰 VALOR TOTAL GASTO: R$ 27,323,316.33
📊 TOTAL DE DESPESAS: 16,805
👥 SENADORES ATIVOS: 97
🏢 FORNECEDORES ÚNICOS: 3,344

🔝 DESTAQUES:
   • Senador que mais gastou: LUCAS BARRETO
     Valor: R$ 511,319.78

   • Tipo de despesa mais comum: Passagens aéreas, aquáticas e terrestres nacionais
     Total: R$ 7,423,646.13

   • Mês com maior gasto: Dez
     Valor: R$ 2,787,342.80

   • Média por despesa: R$ 1,625.90


## 🎓 Conclusões e Insights

### 📌 O que descobrimos:

1. **Concentração de Gastos**: Poucos senadores concentram a maior parte dos gastos
2. **Tipos de Despesa**: Passagens aéreas e divulgação são as categorias mais comuns
3. **Sazonalidade**: Há variação nos gastos ao longo do ano
4. **Distribuição**: A maioria das despesas são de valores menores (até R$ 2.000)
5. **Fornecedores**: Companhias aéreas e empresas de comunicação dominam

### 💡 Próximos Passos:
- Comparar com anos anteriores
- Analisar por partido político
- Identificar padrões suspeitos
- Criar dashboard interativo